# 07 — Structure–conduct–performance regressions

The course notes open with the SCP regression of margins on concentration and its central problem: concentration
is an equilibrium outcome, so the regression does not identify the effect of market structure on prices.
This notebook runs the SCP regression on the airline data in three ways:

HHI is computed by owner (Hawaiian counted with Alaska after the merger closed).

1. **Cross-section** (quarter fixed effects only): log fare on the Herfindahl–Hirschman index (HHI) with route controls.
2. **Within route** (route and quarter fixed effects): uses changes in concentration on a route over time.
3. **Merger-induced concentration as an instrument** (Dafny, Duggan and Ramanarayanan 2012): the Alaska–Hawaiian
   merger raised HHI mechanically by $2 s_{AS} s_{HA}$ on overlap routes. Its interaction with the post-merger
   period instruments HHI. Sample: western routes (overlap routes and routes without either carrier).

Finally, the structural Lerner indices of notebook 03 are related to HHI to show what the structural model implies
about the concentration–margin relationship.

In [ ]:
%run 00_setup.ipynb

In [ ]:
from linearmodels.iv.absorbing import AbsorbingLS
from linearmodels.iv import IV2SLS
p = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")

def market_panel(p):
    g = p.groupby("market_ids")
    # concentration by owner: Hawaiian counts with Alaska from 2024Q4 (the deal closed 18 September 2024)
    owner = np.where((p.carrier == "HA") & (p.t >= qindex(2024, 4)), "AS", p.carrier)
    car = p.assign(owner=owner).groupby(["market_ids", "owner"]).passengers.sum()
    sh = car / car.groupby(level=0).transform("sum")
    m = pd.DataFrame({
        "route": g.route.first(), "quarter": g.quarter.first(), "t": g.t.first(),
        "lfare": g.apply(lambda d: np.log(np.average(d.fare, weights=d.passengers))),
        "hhi": (sh ** 2).groupby(level=0).sum(),
        "n_carriers": g.n_carriers.first(), "log_distance": np.log(g.route_distance.first()),
        "log_pop": g.apply(lambda d: 0.5 * np.log(d.pop_a.iat[0] * d.pop_b.iat[0])),
        "passengers": g.passengers.sum(), "a": g.a.first(), "b": g.b.first()})
    return m.reset_index()

m = market_panel(p)
print(f"{len(m):,} route-quarters; mean HHI {m.hhi.mean():.3f}")
cl = m.route.astype("category").cat.codes

## 1–2. OLS: cross-section and within route

In [ ]:
ols_cs = AbsorbingLS(m.lfare, m[["hhi", "log_distance", "log_pop"]], absorb=pd.DataFrame({"q": m.quarter.astype("category")}),
                     weights=m.passengers).fit(cov_type="clustered", clusters=cl)
ols_fe = AbsorbingLS(m.lfare, m[["hhi"]], absorb=pd.DataFrame({"r": m.route.astype("category"), "q": m.quarter.astype("category")}),
                     weights=m.passengers).fit(cov_type="clustered", clusters=cl)
print(ols_cs.params.round(4).to_dict(), ols_fe.params.round(4).to_dict())

## 3. Merger-induced change in HHI as an instrument

In [ ]:
apt_state = pd.read_parquet(CLEAN / "airport_cbsa.parquet").set_index("AIRPORT").AIRPORT_STATE_CODE
WEST = {"WA", "OR", "CA", "NV", "AZ", "ID", "UT", "MT", "WY", "CO", "NM", "AK", "HI"}
BASE = {qlabel(*q) for q in [(2023, 4), (2024, 1), (2024, 2)]}
base = p[p.quarter.isin(BASE)]
tot = base.groupby("route").passengers.sum()
s_as = base[base.carrier == "AS"].groupby("route").passengers.sum().reindex(tot.index).fillna(0) / tot
s_ha = base[base.carrier == "HA"].groupby("route").passengers.sum().reindex(tot.index).fillna(0) / tot
dhhi = (2 * s_as * s_ha).rename("dhhi_sim")
party_routes = set(p.loc[p.carrier.isin(["AS", "HA"]), "route"])
w = m[m.quarter != "2024Q3"].copy()
w["dhhi_sim"] = w.route.map(dhhi).fillna(0)
w = w[(w.dhhi_sim > 0) | ((~w.route.isin(party_routes)) & (w.a.map(apt_state).isin(WEST) | w.b.map(apt_state).isin(WEST)))]
w["post"] = (w.t >= qindex(2024, 4)).astype(float)
w["z"] = w.dhhi_sim * w.post
fe = pd.DataFrame({"r": w.route.astype("category"), "q": w.quarter.astype("category")})
clw = w.route.astype("category").cat.codes
fs = AbsorbingLS(w.hhi, w[["z"]], absorb=fe, weights=w.passengers).fit(cov_type="clustered", clusters=clw)
rf = AbsorbingLS(w.lfare, w[["z"]], absorb=fe, weights=w.passengers).fit(cov_type="clustered", clusters=clw)
# 2SLS with fixed effects: partial out route and quarter effects (weighted), then IV on the residuals
def wdemean(df, groups, weights, iters=200):
    X = df.astype(float).copy()
    for _ in range(iters):
        X0 = X.copy()
        for g in groups:
            X = X - X.mul(weights, axis=0).groupby(g.values).transform("sum").div(weights.groupby(g.values).transform("sum"), axis=0)
        if (X - X0).abs().max().max() < 1e-10:
            break
    return X
R = wdemean(w[["lfare", "hhi", "z"]], [w.route, w.quarter], w.passengers)
iv = IV2SLS(R.lfare, None, R[["hhi"]], R[["z"]], weights=w.passengers).fit(cov_type="clustered", clusters=clw)
ols_w = AbsorbingLS(w.lfare, w[["hhi"]], absorb=fe, weights=w.passengers).fit(cov_type="clustered", clusters=clw)
print("first stage", fs.params["z"], fs.tstats["z"], "| reduced form", rf.params["z"], "| 2SLS", iv.params["hhi"], iv.std_errors["hhi"])

## Structural markups and concentration

In [ ]:
post_rc = INTERIM / "post_rc_demand_supply.parquet"
pe = pd.read_parquet(post_rc) if post_rc.exists() else None
if pe is not None:
    lm = pe.groupby("market_ids").apply(lambda d: np.average(d.lerner.clip(-1, 1), weights=d.passengers)).rename("lerner")
    mm = m.set_index("market_ids").join(lm).reset_index()
    st = AbsorbingLS(mm.lerner, mm[["hhi"]], absorb=pd.DataFrame({"r": mm.route.astype("category"), "q": mm.quarter.astype("category")}),
                     weights=mm.passengers).fit(cov_type="clustered", clusters=mm.route.astype("category").cat.codes)

In [ ]:
rows = [
    {"specification": "OLS cross-section (quarter FE, route controls)", "dependent": "log fare", "coef on HHI": ols_cs.params["hhi"], "s.e.": ols_cs.std_errors["hhi"], "N": int(ols_cs.nobs)},
    {"specification": "OLS within route (route + quarter FE)", "dependent": "log fare", "coef on HHI": ols_fe.params["hhi"], "s.e.": ols_fe.std_errors["hhi"], "N": int(ols_fe.nobs)},
    {"specification": "OLS within route, western sample", "dependent": "log fare", "coef on HHI": ols_w.params["hhi"], "s.e.": ols_w.std_errors["hhi"], "N": int(ols_w.nobs)},
    {"specification": "First stage: HHI on simulated dHHI x post", "dependent": "HHI", "coef on HHI": fs.params["z"], "s.e.": fs.std_errors["z"], "N": int(fs.nobs)},
    {"specification": "Reduced form: log fare on simulated dHHI x post", "dependent": "log fare", "coef on HHI": rf.params["z"], "s.e.": rf.std_errors["z"], "N": int(rf.nobs)},
    {"specification": "2SLS: HHI instrumented by simulated dHHI x post", "dependent": "log fare", "coef on HHI": iv.params["hhi"], "s.e.": iv.std_errors["hhi"], "N": int(iv.nobs)},
]
if pe is not None:
    rows.append({"specification": "Structural Lerner (RC) on HHI, route + quarter FE", "dependent": "Lerner index", "coef on HHI": st.params["hhi"], "s.e.": st.std_errors["hhi"], "N": int(st.nobs)})
scp = pd.DataFrame(rows).set_index("specification")
save_table(scp, "scp", float_format="%.3f",
           note="Route-quarter observations weighted by passengers; route-clustered standard errors. HHI from carrier passenger shares "
                "(0--1). Simulated dHHI $=2s_{AS}s_{HA}$ from 2023Q4--2024Q2 shares; post = 2024Q4--2025Q2; 2024Q3 excluded.")
scp

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.3))
m["hbin"] = pd.qcut(m.hhi, 20, duplicates="drop")
b = m.groupby("hbin", observed=True).apply(lambda d: pd.Series({"hhi": np.average(d.hhi, weights=d.passengers),
                                                                "lfare": np.average(d.lfare, weights=d.passengers)}))
ax.scatter(b.hhi, np.exp(b.lfare), color=PALETTE[0])
ax.set_xlabel("HHI (carrier passenger shares)"); ax.set_ylabel("Mean fare (2025Q2 $)")
fig.tight_layout(); savefig(fig, "scp_binscatter")
save_json({"scp": scp.reset_index().to_dict("records")}, "scp")